# FraudSpotter: Explainable Fake News Classification
## Google Colab T4 GPU Execution Notebook

This notebook handles all GPU-intensive tasks for FraudSpotter:
1. Direct remote dataset downloading (WELFake, ISOT, LIAR) - **Zero laptop upload required**
2. Classical ML baselines (MultinomialNB, Logistic Regression, Linear SVM)
3. DistilBERT fine-tuning and Optuna hyperparameter optimization
4. Cross-dataset generalization evaluation (WELFake vs ISOT vs LIAR)
5. Explainability (Captum Integrated Gradients + LIME + Faithfulness)
6. ONNX INT8 export and latency benchmarking
7. Exporting lightweight artifacts back to Google Drive

### Step 1: GPU Verification & Google Drive Mount

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

# Set persistent output directory in Drive
import os
DRIVE_ARTIFACTS = "/content/drive/MyDrive/FraudSpotter_Artifacts"
os.makedirs(DRIVE_ARTIFACTS, exist_ok=True)
print(f"Persistent artifact storage ready at: {DRIVE_ARTIFACTS}")

### Step 2: Clone Repository & Install Colab Requirements

In [ ]:
# Replace with your GitHub repository URL or navigate to existing workspace
# !git clone https://github.com/<your-username>/fraudspotter.git /content/fraudspotter
# %cd /content/fraudspotter

!pip install --upgrade pip
!pip install -r requirements-colab.txt

### Step 3: Direct Remote Dataset Acquisition (No Local Upload)
WELFake is downloaded directly from Zenodo. ISOT and LIAR can also be retrieved directly.

In [ ]:
import urllib.request
from pathlib import Path

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

welfake_file = raw_dir / "welfake.csv"
if not welfake_file.exists():
    print("Downloading WELFake directly from Zenodo (~110MB compressed)...")
    zenodo_url = "https://zenodo.org/record/4561253/files/WELFake_Dataset.csv?download=1"
    urllib.request.urlretrieve(zenodo_url, str(welfake_file))
    print(f"WELFake dataset downloaded to {welfake_file}")
else:
    print("WELFake dataset already present.")

### Step 4: Preprocessing and Stratified Splitting

In [ ]:
!python scripts/preprocess_data.py --input_file data/raw/welfake.csv --output_dir data/processed/welfake

### Step 5: Classical Baselines Training
Trains Multinomial Naive Bayes, TF-IDF Logistic Regression, and Linear SVM.

In [ ]:
!python scripts/train_classical.py --model_type all

### Step 6: DistilBERT Fine-Tuning on T4 GPU

In [ ]:
!python scripts/train_transformer.py \
    --train_file data/processed/welfake/train.csv \
    --val_file data/processed/welfake/val.csv \
    --output_dir artifacts/models/distilbert

### Step 7: Comprehensive Evaluation (Macro-F1, Calibration, Bootstrap CIs, McNemar)

In [ ]:
!python scripts/evaluate_models.py \
    --test_file data/processed/welfake/test.csv \
    --output_file artifacts/metrics/evaluation_report.json

import json
with open("artifacts/metrics/evaluation_report.json") as f:
    print(json.dumps(json.load(f), indent=2))

### Step 8: Explainability (Captum Integrated Gradients & LIME)

In [ ]:
!python scripts/explain_predictions.py \
    --model_path artifacts/models/classical/logistic_regression.joblib \
    --sample_text "Breaking: Unverified sources report miraculous medical remedy discovered."

### Step 9: ONNX INT8 Quantization & Latency Benchmark

In [ ]:
!python scripts/export_onnx.py \
    --model_dir artifacts/models/distilbert/best_model \
    --output_onnx artifacts/models/distilbert_onnx/model.onnx \
    --output_quantized artifacts/models/distilbert_onnx/model_quantized.onnx

### Step 10: Synchronize Small Artifacts to Google Drive
Copies only lightweight final models (<150MB INT8 ONNX, metrics JSON, figures) to Drive for easy local download.

In [ ]:
!cp -r artifacts/metrics/* "$DRIVE_ARTIFACTS/metrics/" 2>/dev/null || true
!cp -r artifacts/figures/* "$DRIVE_ARTIFACTS/figures/" 2>/dev/null || true
!cp -r artifacts/models/distilbert_onnx "$DRIVE_ARTIFACTS/" 2>/dev/null || true
!cp -r artifacts/models/classical "$DRIVE_ARTIFACTS/" 2>/dev/null || true
print(f"Artifact synchronization complete. Check {DRIVE_ARTIFACTS} in Google Drive.")